In [1]:
import sys
import os

src_path = os.path.abspath("../src")
sys.path.append(src_path)

from lingo_parser.parser import *
from lingo_parser.transformer import *


from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *

In [81]:
from lark import Transformer, Tree, Token

class LingoModelTransformer3(Transformer):
    
    # --- LE FIX EST ICI : DÉBALLER LES STATEMENTS ---
    def statement(self, items):
        # La grammaire dit : statement: set_block | data_block ...
        # items est une liste contenant un seul élément (le dictionnaire retourné par l'enfant)
        return items[0]
    # ------------------------------------------------

    def simple_name_list(self, items):
        return [str(tok) for tok in items if isinstance(tok, Token) and tok.type == "NAME"]

    def model_decl(self, items):
        return {"model": str(items[0])}

    def end_decl(self, items):
        return None

    def start(self, items):
        model = {"sets": [], "data": {}, "constraints": [], "for_loops": [], "objective": None, "label": None}
        
        # --- DEBUG : Voir ce que le Transformer reçoit à la racine ---
        # print("DEBUG START ITEMS:", items) 
        # -----------------------------------------------------------

        for it in items:
            if it is None or isinstance(it, Token): continue
            
            # Si statement() n'est pas défini, 'it' est un Tree('statement', [...])
            # Avec statement() défini, 'it' est un dict (ex: {'sets': [...]})
            
            if isinstance(it, dict):
                if "sets" in it: 
                    model["sets"].extend(it["sets"]) # Utiliser extend pour fusionner les listes
                elif "data" in it: 
                    model["data"].update(it["data"])
                elif "objective" in it: 
                    model["objective"] = it["objective"]
                elif "label" in it: 
                    model["label"] = it["label"]
                elif "constraint" in it: 
                    model["constraints"].append(it["constraint"])
                elif "for_loop" in it: 
                    model["for_loops"].append(it["for_loop"])
            
            # Cas de secours si 'statement' n'est pas déballé (Tree)
            elif isinstance(it, Tree) and it.data == "statement":
                # On regarde l'enfant direct si le déballage a échoué
                child = it.children[0]
                if isinstance(child, dict):
                    # On refait la logique d'assignation (un peu sale, mais robuste)
                    if "sets" in child: model["sets"].extend(child["sets"])
                    elif "data" in child: model["data"].update(child["data"])
                    # ... etc (mais mieux vaut avoir la méthode statement définie)

        return model

    # --- GESTION DES SETS ---
    def set_block(self, items):
        sets = []
        for item in items:
            if isinstance(item, dict) and "set_decl" in item:
                sets.append(item["set_decl"])
        return {"sets": sets}
    
    def set_decl(self, items):
        name = str(items[0])
        
        # Cas ARC(MACHINES,PRODUITS):...
        if len(items) > 1 and isinstance(items[1], Token) and items[1].type == "LPAR":
            indices = []
            i = 2
            while i < len(items):
                tok = items[i]
                if isinstance(tok, Token) and tok.type == "RPAR": break
                if isinstance(tok, Token) and tok.type == "NAME": indices.append(str(tok))
                i += 1
            
            attrs = []
            for it in items:
                if isinstance(it, dict) and "name_list" in it: attrs = it["name_list"]
            
            return {"set_decl": {"name": name, "indices": indices, "attrs": attrs}}
        
        # Cas classique MACHINES/1,2/:...
        else:
            elements = []
            attrs = []
            for it in items:
                if isinstance(it, dict) and "name_list" in it:
                    if not elements: elements = it["name_list"]
                    else: attrs = it["name_list"]
            return {"set_decl": {"name": name, "elements": elements, "attrs": attrs}}

    def name_list(self, items):
        return {"name_list": [str(tok) for tok in items if isinstance(tok, Token) and tok.type in ("NAME", "NUMBER")]}

    # --- GESTION DES DATA ---
    def data_block(self, items):
        data = {}
        for item in items:
            if isinstance(item, dict): data.update(item)
        return {"data": data}

    def data_stmt(self, items):
        key = str(items[0])
        def flatten(tree):
            vals = []
            if isinstance(tree, Token) and tree.type == "NUMBER": vals.append(float(tree))
            elif isinstance(tree, Tree):
                for c in tree.children: vals.extend(flatten(c))
            return vals
        values = flatten(items[1])
        return {key: values}

    # --- OBJECTIFS ET CONTRAINTES ---
    def objective(self, items):
        dir_tok = "MAX"
        for it in items:
            if isinstance(it, Token) and str(it).upper() in ("MAX", "MIN"):
                dir_tok = str(it).upper()
        
        expr_tree = next((x for x in items if isinstance(x, Tree)), None)
        expr_str = self._expr_to_str(expr_tree) if expr_tree else "0"
        return {"objective": f"{dir_tok} = {expr_str}"}

    def constraint(self, items):
        filtered = [i for i in items if not (isinstance(i, Token) and i.type == "SEMICOLON")]
        
        if len(filtered) == 1 and isinstance(filtered[0], dict) and "for_loop" in filtered[0]:
            return filtered[0]
        
        if len(filtered) >= 3:
            left = self._expr_to_str(filtered[0])
            op = str(filtered[1])
            right = self._expr_to_str(filtered[2])
            return {"constraint": f"{left} {op} {right}"}
        return {"constraint": "ERROR_PARSING"}

    def for_loop(self, items):
        indexset = next((it for it in items if isinstance(it, Tree) and it.data == "indexset"), None)
        expr = next((it for it in items if isinstance(it, Tree) and it.data not in ("indexset", "COLON")), None)
        
        # Fallback pour expr si non trouvé proprement
        if expr is None:
             # On prend le dernier élément qui n'est pas un token de structure
             expr = items[-2] # Juste avant RPAR/SEMICOLON

        idx_str = self._expr_to_str(indexset)
        expr_str = self._expr_to_str(expr)
        return {"for_loop": f"@FOR({idx_str}: {expr_str})"}

    # --- HELPERS ---
    def indexset(self, items): return items[0]
    
    def indexed_set(self, items):
        name = str(items[0])
        indices = items[2] 
        return Tree("indexed_set", [name, indices])

    def param_ref(self, items):
        name = str(items[0])
        indices = items[2] 
        return Tree("param_ref", [name, indices])

    def gin_expr(self, items): return Tree("gin_expr", [items[2]])
    def bin_expr(self, items): return Tree("bin_expr", [items[2]])
    def expr_with_comp(self, items): return Tree("expr_with_comp", items)
    def label(self, items): return {"label": str(items[0])}
    def comp_op(self, items): return str(items[0])

    def _expr_to_str(self, tree):
        if isinstance(tree, Token): return str(tree)
        if isinstance(tree, list): return ",".join(tree)
        if not isinstance(tree, Tree): return str(tree)

        data = tree.data
        if data in ("indexed_set", "param_ref"):
            name = str(tree.children[0])
            indices = tree.children[1]
            return f"{name}({','.join(indices)})"
            
        if data == "expr_with_comp":
            left = self._expr_to_str(tree.children[0])
            op = str(tree.children[1])
            right = self._expr_to_str(tree.children[2])
            return f"{left} {op} {right}"

        if data == "gin_expr": return f"@GIN({self._expr_to_str(tree.children[0])})"
        if data == "bin_expr": return f"@BIN({self._expr_to_str(tree.children[0])})"

        if data == "sum_expr":
            idx_node = next((c for c in tree.children if isinstance(c, Tree) and c.data in ("indexset", "indexed_set")), None)
            # On cherche l'expression après les deux points
            # Astuce: on prend tous les enfants après COLON
            found_colon = False
            expr_parts = []
            for c in tree.children:
                if isinstance(c, Token) and c.type == "COLON": found_colon = True; continue
                if found_colon and isinstance(c, Token) and c.type == "RPAR": break
                if found_colon: expr_parts.append(c)
            
            # Si on a trouvé des parts, on les convertit
            if expr_parts:
                expr_str = "".join([self._expr_to_str(p) for p in expr_parts])
            else:
                # Fallback ancien
                expr_node = next((c for c in tree.children if isinstance(c, Tree) and c.data in ("expr", "term", "factor")), None)
                expr_str = self._expr_to_str(expr_node) if expr_node else "?"

            idx_str = self._expr_to_str(idx_node) if idx_node else "?"
            return f"@SUM({idx_str}:{expr_str})"

        parts = []
        for c in tree.children:
            val = self._expr_to_str(c)
            parts.append(val)
        return "".join(parts)

In [5]:
tree = parse_lingo_model("../data/cardoza.lng")
model_dict = LingoModelTransformer2().transform(tree)
pyomo_code = generate_pyomo_code(model_dict)
print(pyomo_code)

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals

{'sets': [{'name': 'VILLES',
   'elements': ['LA', 'SF', 'SD'],
   'attrs': ['gain_usine',
    'gain_magasin',
    'capital_usine',
    'capital_magasin',
    'choix_usine',
    'choix_magasin']}],
 'data': {'gain_usine': [9.0, 5.0, 7.0],
  'gain_magasin': [6.0, 4.0, 5.0],
  'capital_usine': [6.0, 3.0, 4.0],
  'capital_magasin': [5.0, 2.0, 3.0],
  'Dispo': [10.0]},
 'constraints': ['@SUM(VILLES(v): capital_magasin ( v ) * choix_magasin ( v ) + capital_usine ( v ) * choix_usine ( v )) <= Dispo',
  '@SUM(VILLES(v): choix_magasin ( v )) >= 1',
  '@SUM(VILLES(v): choix_usine ( v )) >= 1'],
 'for_loops': ['@FOR(VILLES(v): choix_magasin ( v ) <= choix_usine ( v ))',
  '@FOR(VILLES(v): @BIN(())',
  '@FOR(VILLES(v): @BIN(())'],
 'objective': 'MAX = @SUM(VILLES(v): gain_usine ( v ) * choix_usine ( v ) + gain_magasin ( v ) * choix_magasin ( v ))',
 'label': '['}

In [6]:
model_dict

{'sets': [{'name': 'FLEURS',
   'elements': ['Lys', 'Roses', 'Jonquilles'],
   'attrs': ['Dispo']},
  {'name': 'BOUQUETS', 'elements': ['B1', 'B2'], 'attrs': ['Prix', 'X']},
  {'name': 'ARCS', 'indices': ['FLEURS', 'BOUQUETS'], 'attrs': ['Compo']}],
 'data': {'Dispo': [50.0, 80.0, 80.0],
  'Prix': [40.0, 50.0],
  'Compo': [10.0, 10.0, 10.0, 20.0, 20.0, 10.0]},
 'constraints': [],
 'for_loops': ['@FOR(FLEURS(f): @SUM(BOUQUETS(b): Compo(f,b) * X ( b )) <= Dispo ( f ))'],
 'objective': 'MAX = @SUM(BOUQUETS: Prix * X)',
 'label': '['}

In [3]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.FLEURS = Set(initialize=['Lys', 'Roses', 'Jonquilles'])
model.BOUQUETS = Set(initialize=['B1', 'B2'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.FLEURS for j in model.BOUQUETS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Dispo = Param(model.FLEURS, initialize={'Lys': 50.0, 'Roses': 80.0, 'Jonquilles': 80.0}, within=NonNegativeReals)
model.Prix = Param(model.BOUQUETS, initialize={'B1': 40.0, 'B2': 50.0}, within=NonNegativeReals)
model.Compo = Param(model.FLEURS, model.BOUQUETS, initialize={('Lys', 'B1'): 10.0, ('Lys', 'B2'): 10.0, ('Roses', 'B1'): 10.0, ('Roses', 'B2'): 20.0, ('Jonquilles', 'B1'): 20.0, ('Jonquilles', 'B2'): 10.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.X = Var(model.BOUQUETS, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c_for_0 = ConstraintList()
for f in model.FLEURS:
    model.c_for_0.add(sum(model.Compo[f,b] * model.X[b] for b in model.BOUQUETS) <= model.Dispo[f])

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(model.Prix[b] * model.X[b] for b in model.BOUQUETS), sense=maximize)

In [4]:


# === Résolution ===
solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')

Read LP format model from file C:\Users\joaqu\AppData\Local\Temp\tmpnpmon0vu.pyomo.lp
Reading time = 0.00 seconds
x1: 3 rows, 2 columns, 6 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-14700KF, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 28 logical processors, using up to 28 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model fingerprint: 0x42fed3e5
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+01, 2e+01]
  Objective range  [4e+01, 5e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 8e+01]
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros



Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2500000e+31   3.125000e+30   2.250000e+01      0s
       2    2.3000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.300000000e+02
Variable set: X
   B1 = 2.0
   B2 = 3.0


In [10]:
from pyomo.environ import *

model = ConcreteModel()

model.FLEURS = Set(initialize=['Lys', 'Roses', 'Jonquilles'])
model.BOUQUETS = Set(initialize=['B1', 'B2'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.FLEURS for j in model.BOUQUETS])
model.Dispo = Param(model.FLEURS, initialize={'Lys': 50.0, 'Roses': 80.0, 'Jonquilles': 80.0}, within=NonNegativeReals)
model.Prix = Param(model.BOUQUETS, initialize={'B1': 40.0, 'B2': 50.0}, within=NonNegativeReals)
model.Compo = Param(model.FLEURS, model.BOUQUETS, initialize={('Lys', 'B1'): 10.0, ('Lys', 'B2'): 10.0, ('Roses', 'B1'): 10.0, ('Roses', 'B2'): 20.0, ('Jonquilles', 'B1'): 20.0, ('Jonquilles', 'B2'): 10.0}, within=NonNegativeReals)
model.X = Var(model.BOUQUETS, domain=NonNegativeReals)
def rule_for_0(model, f):
    return sum(model.Compo[f,b] * model.X[b] for b in model.BOUQUETS) <= model.Dispo[f]
model.c_for_0 = Constraint(model.FLEURS, rule=rule_for_0)
model.obj = Objective(expr=sum(model.Prix[b] * model.X[b] for b in model.BOUQUETS), sense=maximize)

solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

Read LP format model from file /var/folders/9m/3q_5vd254tn5nc58zx_4fpth0000gn/T/tmpd8j_xspx.pyomo.lp
Reading time = 0.00 seconds
x1: 3 rows, 2 columns, 6 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros
Model fingerprint: 0x42fed3e5
Coefficient statistics:
  Matrix range     [1e+01, 2e+01]
  Objective range  [4e+01, 5e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 8e+01]
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2500000e+31   3.125000e+30   2.250000e+01      0s
       2    2.3000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.300000000e+02


{'Problem': [{'Name': 'x1', 'Lower bound': 230.0, 'Upper bound': 230.0, 'Number of objectives': 1, 'Number of constraints': 3, 'Number of variables': 2, 'Number of binary variables': 0, 'Number of integer variables': 0, 'Number of continuous variables': 2, 'Number of nonzeros': 6, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Return code': 0, 'Message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Termination condition': 'optimal', 'Termination message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Wall time': 0.0009410381317138672, 'Error rc': 0}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

In [11]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.x = Var(model.PRODUITS, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c_for_0 = ConstraintList()
for m in model.MACHINES:
    model.c_for_0.add(sum(model.temps[m,p] * model.x[p] for a in model.ARC) <= model.disponibilite[m])

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(gain * x for p in model.PRODUITS), sense=maximize)

solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

NameError: name 'p' is not defined

In [5]:


def generate_notebook() :

    tree = parse_lingo_model("../data/fleuriste_alg.lng")
    model_dict = LingoModelTransformer2().transform(tree)

    pyomo_code = generate_pyomo_code(model_dict)

    generate_pyomo_notebook(pyomo_code, solver="gurobi", filename="fleuristeaaa.ipynb")

generate_notebook()

✅ Notebook généré : fleuristeaaa.ipynb


In [1]:
!pyomo help --solvers


Pyomo Solvers and Solver Managers
---------------------------------
Pyomo uses 'solver managers' to execute 'solvers' that perform
optimization and other forms of model analysis.  A solver directly
executes an optimizer, typically using an executable found on the
user's PATH environment.  Solver managers support a flexible mechanism
for asynchronously executing solvers either locally or remotely.  The
following solver managers are available in Pyomo:

    neos       Asynchronously execute solvers on the NEOS server
    serial     Synchronously execute solvers locally

If no solver manager is specified, Pyomo uses the serial solver
manager to execute solvers locally.  The neos solver manager is used
to execute solvers on the NEOS optimization server.


Serial Solver Interfaces
------------------------
The serial manager supports the following solver interfaces:

    appsi_cbc                    Automated persistent interface to Cbc
    appsi_cplex                  Automated persistent 